In [14]:
import numpy as np 
import pandas as pd

In [15]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, weights=[0.9, 0.1], flip_y=0, random_state=42)

In [16]:
np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# model 1

In [18]:
from sklearn.linear_model import LogisticRegression

# Train the model
log_reg = LogisticRegression(C=1, solver='liblinear')
log_reg.fit(X_train, y_train)

# Predict on the test set
pred = log_reg.predict(X_test)

In [19]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



# model 2

In [20]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators= 30, max_depth=3)

rf_clf.fit(X_train, y_train)

pred = rf_clf.predict(X_test)

In [21]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98       270
           1       0.95      0.70      0.81        30

    accuracy                           0.97       300
   macro avg       0.96      0.85      0.89       300
weighted avg       0.97      0.97      0.96       300



# model 3

In [22]:
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

xgb_clf.fit(X_train, y_train)

pred = xgb_clf.predict(X_test)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:59:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [23]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



# model 4

imbalance classification

In [24]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)

smt_X_train, smt_y_train = smt.fit_resample(X_train, y_train)

In [25]:
np.unique(smt_y_train, return_counts=True)

(array([0, 1]), array([619, 619]))

In [26]:
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

xgb_clf.fit(smt_X_train, smt_y_train)

pred = xgb_clf.predict(X_test)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:59:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [27]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



# mlflow 1

## Set the Experiment

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.combine import SMOTETomek

In [29]:
models = [
    (
        "Logistic Regression",
        LogisticRegression(C=1, solver='liblinear'),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest",
        RandomForestClassifier(n_estimators=30, max_depth=3),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        (smt_X_train, smt_y_train),
        (X_test, y_test)
    )
]

In [30]:
reports = []

for model_name, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    
    X_test = test_set[0]
    y_test = test_set[1]

    model.fit(X_train, y_train)
    
    pred = model.predict(X_test)
    
    report = classification_report(y_test, pred, output_dict=True)
    reports.append(report)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:59:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:59:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [72]:
# reports

In [70]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

mlflow.set_experiment("Second Experiment")
mlflow.set_tracking_uri(uri='http://127.0.0.1:5000/')

for i, element in enumerate(models):
    model_name = element[0]

    model = element[1]

    report = reports[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_param('model', model_name)
        mlflow.log_metric('accuracy', report['accuracy'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('f1_score_macro', report['macro avg']['f1-score'])        
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")          


2026/03/11 10:57:43 INFO mlflow.tracking.fluent: Experiment with name 'Second Experiment' does not exist. Creating a new experiment.
2026/03/11 10:57:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/11 10:57:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/6/runs/11790a5e1d554ee39e947f93f6de4021
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6


2026/03/11 10:58:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/11 10:58:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/6/runs/4bec97f701fb490191645f74061f3eed
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6


2026/03/11 10:58:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/6/runs/96b53e35ed7948749db59cdfcee63972
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6


2026/03/11 10:58:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/6/runs/1f82bd1840a0420e941d25f728f202ca
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6


## Register the model

In [37]:
run_id = '1f82bd1840a0420e941d25f728f202ca'

model_name = "XGB-Smote"

model_uri = f"runs:/{run_id}/model"

result = mlflow.register_model(model_uri, model_name)

Successfully registered model 'XGB-Smote'.
2026/03/13 14:29:53 WARNING mlflow.tracking._model_registry.fluent: Run with id 1f82bd1840a0420e941d25f728f202ca has no artifacts at artifact path 'model', registering model based on models:/m-b2e230e4a22045fca86a468feba7f620 instead
2026/03/13 14:29:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1
Created version '1' of model 'XGB-Smote'.


### Load the model from register

In [ ]:
import mlflow.xgboost

model_name = "XGB-Smote"

model_version = 1

model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)

In [ ]:
# (or) with alias name

# model_name = "XGB-Smote"

# model_version = 1

# model_uri = f"models:/{model_name}@challenger"

# loaded_model = mlflow.xgboost.load_model(model_uri)

In [47]:
y_pred = loaded_model.predict(smt_X_train)

y_pred[:4]

array([0, 0, 1, 0])

## Production

In [ ]:
model_name = "XGB-Smote"

# model_version = 1

dev_model_uri = f"models:/{model_name}@challenger"

prod_model = 'anomaly-detection-prod'

client = mlflow.MlflowClient()

client.copy_model_version(dev_model_uri, prod_model)

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'XGB-Smote' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1773393148146, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1773393148146, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='1f82bd1840a0420e941d25f728f202ca', run_link='', source='models:/XGB-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

### Load the model from production

In [50]:
import mlflow.xgboost

prod_model = 'anomaly-detection-prod'

model_uri = f"models:/{prod_model}@champion"

loaded_model = mlflow.xgboost.load_model(model_uri)

In [53]:
y_pred = loaded_model.predict(smt_X_train)

y_pred[:4]

array([0, 0, 1, 0])

## Extra

In [51]:
run_id = '11790a5e1d554ee39e947f93f6de4021'

model_name = "Logistic Regression"

model_uri = f"runs:/{run_id}/model"

result = mlflow.register_model(model_uri, model_name)

Successfully registered model 'Logistic Regression'.
2026/03/13 14:51:53 WARNING mlflow.tracking._model_registry.fluent: Run with id 11790a5e1d554ee39e947f93f6de4021 has no artifacts at artifact path 'model', registering model based on models:/m-4e20c4da41bc46a8b22a8f2fa6e180e9 instead
2026/03/13 14:51:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Logistic Regression, version 1
Created version '1' of model 'Logistic Regression'.


In [58]:
import mlflow.sklearn

model_name = "Logistic Regression"

model_version = 1

model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.sklearn.load_model(model_uri)

In [62]:
y_pred = loaded_model.predict(X_test)

y_pred[:4]

array([0, 0, 0, 0])

In [64]:
model_name = "Logistic Regression"

# model_version = 1

dev_model_uri = f"models:/{model_name}@challenger"

prod_model = 'delete'

client = mlflow.MlflowClient()

client.copy_model_version(dev_model_uri, prod_model)

Successfully registered model 'delete'.
Copied version '1' of model 'Logistic Regression' to version '1' of model 'delete'.


<ModelVersion: aliases=[], creation_timestamp=1773393926262, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1773393926262, metrics=None, model_id=None, name='delete', params=None, run_id='11790a5e1d554ee39e947f93f6de4021', run_link='', source='models:/Logistic Regression/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [65]:
import mlflow.sklearn

prod_model = 'delete'

model_uri = f"models:/{prod_model}@champion"

loaded_model = mlflow.sklearn.load_model(model_uri)

In [66]:
y_pred = loaded_model.predict(X_test)

y_pred[:4]

array([0, 0, 0, 0])

# DagsHub

In [68]:
# pip install dagshub

In [70]:
import dagshub
dagshub.init(repo_owner='haribabuofficial', repo_name='mlflow_dagshub_youtube', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=559833c1-21ef-423d-9676-8fa57f643d2e&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=2ee6ed15c7b93364809c5bc2d26d023257fa43a9b12902e3fa7709d600aa8e46




Accessing as haribabuofficial

Initialized MLflow to track repo "haribabuofficial/mlflow_dagshub_youtube"

Repository haribabuofficial/mlflow_dagshub_youtube initialized!

In [71]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

mlflow.set_experiment("Second Experiment")
# mlflow.set_tracking_uri(uri='http://127.0.0.1:5000/')

for i, element in enumerate(models):
    model_name = element[0]

    model = element[1]

    report = reports[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_param('model', model_name)
        mlflow.log_metric('accuracy', report['accuracy'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('f1_score_macro', report['macro avg']['f1-score'])        
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")          


2026/03/13 15:31:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 15:31:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 15:31:24 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/6/runs/26b70e0252074a73870e0d11cf3b0dcc
🧪 View experiment at: http://localhost:5000/#/experiments/6


2026/03/13 15:31:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 15:31:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 15:31:37 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


🏃 View run Random Forest at: http://localhost:5000/#/experiments/6/runs/c3b0722ec0fa4ac6b69da722ea8f26f0
🧪 View experiment at: http://localhost:5000/#/experiments/6


2026/03/13 15:31:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/6/runs/b774c24b0c2c4880b8d79b61c3654267
🧪 View experiment at: http://localhost:5000/#/experiments/6


AttributeError: 'dict' object has no attribute 'save_model'

# mlflow 2

In [34]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (smt_X_train, smt_y_train),
        (X_test, y_test)
    )
]

In [35]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:02:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:02:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [36]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

mlflow.set_experiment("delete")
mlflow.set_tracking_uri("http://localhost:5000")

for i, element in enumerate(models):
    model_name = element[0]

    model = element[1]
    
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):  
        mlflow.log_params(params)      
        mlflow.log_param("model", model_name)
        mlflow.log_metric('accuracy', report['accuracy'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('f1_score_macro', report['macro avg']['f1-score'])        
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  

2026/03/13 12:02:45 INFO mlflow.tracking.fluent: Experiment with name 'delete' does not exist. Creating a new experiment.
2026/03/13 12:02:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 12:02:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 12:02:45 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/8/runs/5e53b61ae25e4df69ef053ab649e981b
🧪 View experiment at: http://localhost:5000/#/experiments/8


2026/03/13 12:03:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 12:03:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 12:03:03 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


🏃 View run Random Forest at: http://localhost:5000/#/experiments/8/runs/fdadc91e558d42db84a17de4ea987d0a
🧪 View experiment at: http://localhost:5000/#/experiments/8


2026/03/13 12:03:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/8/runs/c78fe6edbf754a1db6a070dec4aba001
🧪 View experiment at: http://localhost:5000/#/experiments/8


AttributeError: 'dict' object has no attribute 'save_model'

# Register the model

In [ ]:
run_id = input("Enter Run ID: ")

model_name = "XGB-Smote"

model_uri = f"runs:/{run_id}/model"

result = mlflow.register_model(model_uri, model_name)

Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/03/11 16:57:51 WARNING mlflow.tracking._model_registry.fluent: Run with id 1f82bd1840a0420e941d25f728f202ca has no artifacts at artifact path 'model', registering model based on models:/m-b2e230e4a22045fca86a468feba7f620 instead
2026/03/11 16:57:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1
Created version '1' of model 'XGB-Smote'.


# Load the model

In [84]:
model_version = 1

model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)

y_pred = loaded_model.predict(X_test)

In [ ]:
# production

# dev_model_uri = f"models:/{model_name}/{model_version}"

# prod_model = 'anomaly-detection-prod'

# client = mlflow.MlflowClient()

# client.copy_model_version(src_model_uri='', dst_name='')